<a href="https://colab.research.google.com/github/saksaw/AI-Model-Evaluation-and-Data-Quality-Suite/blob/main/Multimodal_%26_Image_Data_Quality_Audit_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

print("🚀 Initializing Automated Multimodal Data Quality & Alignment Pipeline...")

# =====================================================================
# 1. THE MULTIMODAL EVALUATION DATASET
# =====================================================================
# This dataset simulates a production batch where an AI model predicted
# image content based on localized search queries, compared to human ground truth.
multimodal_batch = [
    {
        "query": "running shoes waterproof",
        "image_filename": "img_001.jpg",
        "predicted_tags": ["footwear", "shoes", "outdoor", "waterproof"],
        "human_ground_truth": ["footwear", "shoes", "outdoor", "waterproof"],
        "bounding_box_confidence": 0.94
    },
    {
        "query": "red leather handbag",
        "image_filename": "img_002.jpg",
        "predicted_tags": ["accessory", "backpack", "brown", "synthetic"],
        # MISMATCH DETECTED: The model hallucinated a brown backpack instead of a red leather handbag.
        "human_ground_truth": ["accessory", "handbag", "red", "leather"],
        "bounding_box_confidence": 0.41
    },
    {
        "query": "smartphone tempered glass",
        "image_filename": "img_003.jpg",
        "predicted_tags": ["electronics", "smartphone", "screen-protector"],
        "human_ground_truth": ["electronics", "smartphone", "glass"],
        "bounding_box_confidence": 0.88
    }
]

# =====================================================================
# 2. AUTOMATED VALIDATION FUNCTIONS
# =====================================================================
def calculate_iou_tags(predicted, ground_truth):
    """
    Calculates Intersection over Union (IoU) concept scores for the tag arrays.
    Measures strict semantic overlap between model predictions and human reality.
    """
    set_pred = set(predicted)
    set_gt = set(ground_truth)

    intersection = set_pred.intersection(set_gt)
    union = set_pred.union(set_gt)

    if not union:
        return 0.0
    return len(intersection) / len(union)

# =====================================================================
# 3. RUNNING THE MULTIMODAL MODEL EVALUATION RUN
# =====================================================================
audit_log = []
IOU_THRESHOLD = 0.60
CONFIDENCE_THRESHOLD = 0.70

for item in multimodal_batch:
    # 1. Measure tag alignment score
    tag_alignment_score = calculate_iou_tags(item["predicted_tags"], item["human_ground_truth"])

    # 2. Flag anomalies (Low bounding box confidence or weak tag overlap)
    is_hallucinating = tag_alignment_score < IOU_THRESHOLD
    low_confidence = item["bounding_box_confidence"] < CONFIDENCE_THRESHOLD

    # 3. Assign Human-in-the-Loop (HITL) Routing status based on error flags
    if is_hallucinating or low_confidence:
        status = "FLAGGED: Route to HITL Exception Audit"
    else:
        status = "PASSED: High Alignment verified"

    audit_log.append({
        "Image File": item["image_filename"],
        "Search Query": item["query"],
        "Tag Alignment Score": round(tag_alignment_score, 2),
        "Model Confidence": item["bounding_box_confidence"],
        "Quality Audit Verdict": status
    })

# =====================================================================
# 4. EXPORTING THE AUTOMATED QUALITY AUDIT REPORT
# =====================================================================
df_multimodal_report = pd.DataFrame(audit_log)
print("\n📊 COMPUTER VISION & MULTIMODAL MODEL ALIGNMENT REPORT:")
print("=======================================================================================")
print(df_multimodal_report.to_string(index=False))

🚀 Initializing Automated Multimodal Data Quality & Alignment Pipeline...

📊 COMPUTER VISION & MULTIMODAL MODEL ALIGNMENT REPORT:
 Image File              Search Query  Tag Alignment Score  Model Confidence                  Quality Audit Verdict
img_001.jpg  running shoes waterproof                 1.00              0.94        PASSED: High Alignment verified
img_002.jpg       red leather handbag                 0.14              0.41 FLAGGED: Route to HITL Exception Audit
img_003.jpg smartphone tempered glass                 0.50              0.88 FLAGGED: Route to HITL Exception Audit
